# MT5bot_m4Gold v1.30 â€” Out-of-sample backtest on comprehensive XAUUSD

This notebook runs the **live v1.30 EA's decision logic** (XGBoost meta-gate + 24-feature S/R+Fib) on a long-history XAUUSD dataset to test whether the strategy survives out-of-sample.

**What's being tested**: the same ONNX meta-gate that the live MT5 EA executes, plus the same `aurum.sr_fib_features` Python extractor, plus a faithful Python port of the EA's exit engine (initial SL â†’ breakeven move â†’ ATR trailing stop â†’ trend-flip / timeout / SL hit). The MT5 tick-order simulation is mirrored (bullish bars: lowâ†’high; bearish: highâ†’low) so SL-vs-trail-vs-breakeven race conditions resolve the same way as in MT5 Tester.

**Reference baseline** (from MT5 Tester on 2025-11-20 â†’ 2026-05-21):
- v1.10 18-feat: +11.83%, PF 1.25, 3.4% max DD
- Python backtester on same window: +16.97% â€” ~5pp more optimistic than MT5 due to no per-bar slippage modelling. **Subtract ~5pp from any Kaggle result for honest expectations.**

## How to use
1. Add the Kaggle dataset `feriandanaputra/comprehensive-xauusd-historical-price-data` as input
2. Run all cells
3. Edit the **Parameters** cell to test your hypothesis (date window, lot, spread assumption)

## Cell 1 â€” Install deps + clone repo

In [ ]:
!pip install -q onnxruntime
import os, sys, subprocess
from datetime import datetime, timezone

REPO = '/kaggle/working/mt5bot_m4GOLD'
GIT_URL = 'https://github.com/bongc4947/mt5bot_m4GOLD.git'

# Always force-sync with origin/master so we never run a stale clone.
if os.path.isdir(REPO):
    subprocess.run(['git', '-C', REPO, 'fetch', 'origin', '--depth', '1'],
                   check=True, capture_output=True)
    subprocess.run(['git', '-C', REPO, 'reset', '--hard', 'origin/master'],
                   check=True, capture_output=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', GIT_URL, REPO],
                   check=True, capture_output=True)

# Show the HEAD so the user can sanity-check the clone is the latest push.
head_hash = subprocess.run(
    ['git', '-C', REPO, 'rev-parse', '--short', 'HEAD'],
    capture_output=True, text=True).stdout.strip()
head_ts_unix = int(subprocess.run(
    ['git', '-C', REPO, 'log', '-1', '--format=%ct', 'HEAD'],
    capture_output=True, text=True).stdout.strip())
head_subject = subprocess.run(
    ['git', '-C', REPO, 'log', '-1', '--format=%s', 'HEAD'],
    capture_output=True, text=True).stdout.strip()
head_ts = datetime.fromtimestamp(head_ts_unix, tz=timezone.utc)
now      = datetime.now(timezone.utc)
age_min  = (now - head_ts).total_seconds() / 60.0

# Invalidate cached imports so notebook picks up the freshly-pulled code.
if REPO + '/python' in sys.path:
    sys.path.remove(REPO + '/python')
sys.path.insert(0, f'{REPO}/python')
for mod in list(sys.modules):
    if mod.startswith('kaggle_backtest_ea') or mod.startswith('aurum'):
        del sys.modules[mod]

print('repo synced:        ', REPO)
print(f'HEAD commit:         {head_hash}')
print(f'HEAD subject:        {head_subject}')
print(f'HEAD timestamp:      {head_ts.isoformat()}  ({age_min:.0f} min ago)')
print()
print(f'onnx exists:         {os.path.exists(f"{REPO}/onnx_out/M4GOLD_METATREND_GOLD.onnx")}')
print(f'spec exists:         {os.path.exists(f"{REPO}/onnx_out/M4GOLD_METATREND_GOLD_spec.json")}')


## Cell 2 â€” Locate the Kaggle dataset file

In [ ]:
import os, re
ROOT = '/kaggle/input'
all_files = []
for dirpath, dirs, files in os.walk(ROOT):
    for f in files:
        full = os.path.join(dirpath, f)
        size_mb = os.path.getsize(full) / 1e6
        all_files.append((full, size_mb))
        print(f'  {full}   {size_mb:.1f} MB')

# Auto-detect the best M5 (or M1 for resample) file
def _score(path):
    name = os.path.basename(path).lower()
    xau_bonus = 5 if 'xau' in name else 0
    if re.search(r'(_m5_|_5m_|m5\.|5min)', name): return 100 + xau_bonus
    if re.search(r'(_m1_|_1m_|m1\.|1min)', name): return 80  + xau_bonus
    if re.search(r'(_m15_|15m|15min)',    name): return 50  + xau_bonus
    if re.search(r'(_h1_|1h\.|hourly)',   name): return 20  + xau_bonus
    if name.endswith('.csv') or name.endswith('.parquet'): return 1 + xau_bonus
    return 0
ranked = sorted(all_files, key=lambda p: (-_score(p[0]), -p[1]))
if ranked:
    SUGGESTED_PATH = ranked[0][0]
    print()
    print('>>> SUGGESTED DATA_PATH for Cell 3:')
    print(f"    DATA_PATH = '{SUGGESTED_PATH}'")
else:
    SUGGESTED_PATH = None
    print('NO files found in /kaggle/input - is any dataset attached? Use the right-side "Add Data" panel.')


## Cell 3 â€” Parameters
Edit these and re-run from this point down to test different hypotheses.

In [ ]:
# === EDIT ME ============================================================
# Leave DATA_PATH = None to auto-use the SUGGESTED_PATH from Cell 2.
DATA_PATH    = None
FROM_DATE    = None    # e.g. '2010-01-01' or None for all
TO_DATE      = None    # e.g. '2020-12-31' or None for all
DEPOSIT      = 10000.0
LOT          = 0.01
MAX_STACK    = 1       # 1 = no pyramiding (Tester-validated)

# Cost model: per-bar variable spread (from MT5 HST <SPREAD> column) is the
# most realistic. If the dataset doesn't have a SPREAD column, we fall back
# to a flat round-trip cost of SPREAD_USD in price units.
USE_VARIABLE_SPREAD = True
SPREAD_USD          = 0.05    # fallback flat cost (only used if variable spread unavailable)
# ========================================================================

if DATA_PATH is None:
    try:
        DATA_PATH = SUGGESTED_PATH
    except NameError:
        raise RuntimeError('Run Cell 2 first to discover the dataset path.')
print(f'DATA_PATH           = {DATA_PATH}')
print(f'WINDOW              = {FROM_DATE or "start"} -> {TO_DATE or "end"}')
print(f'DEPOSIT             = ${DEPOSIT}, LOT = {LOT}')
print(f'USE_VARIABLE_SPREAD = {USE_VARIABLE_SPREAD} (fallback {SPREAD_USD})')
print(f'MAX_STACK           = {MAX_STACK}')


## Cell 4 â€” Load + audit the data

In [ ]:
import logging, pandas as pd
from pathlib import Path
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s')
from kaggle_backtest_ea import load_m5_data

if not Path(DATA_PATH).exists():
    msg = f'DATA_PATH does not exist: {DATA_PATH}. Run Cell 2 to list real files and update DATA_PATH in Cell 3.'
    raise FileNotFoundError(msg)

m5 = load_m5_data(Path(DATA_PATH))
if FROM_DATE: m5 = m5[m5['time'] >= pd.Timestamp(FROM_DATE, tz='UTC')]
if TO_DATE:   m5 = m5[m5['time'] <= pd.Timestamp(TO_DATE,   tz='UTC')]
m5 = m5.reset_index(drop=True)
print(f'bars: {len(m5):,}')
print(f'span: {m5["time"].iloc[0]}  ->  {m5["time"].iloc[-1]}')
print(f'gold price range: ${m5["close"].min():.2f}  ->  ${m5["close"].max():.2f}')
m5.head()


## Cell 5 â€” Load the meta-gate ONNX

In [ ]:
from kaggle_backtest_ea import load_meta_gate, DEFAULT_INPUTS, run_backtest
from pathlib import Path
ONNX = Path(REPO) / 'onnx_out' / 'M4GOLD_METATREND_GOLD.onnx'
SPEC = Path(REPO) / 'onnx_out' / 'M4GOLD_METATREND_GOLD_spec.json'
sess, in_name, spec = load_meta_gate(ONNX, SPEC)
print(f'meta-gate version: {spec.get("version")}')
print(f'n_features:        {spec["n_features"]}')
print(f'act_threshold:     {spec["act_threshold"]}')
print(f'CV meanPF:         {spec["cv"]["meta_mean_pf"]}')

## Cell 6 â€” Run the backtest
Takes ~1â€“10 minutes depending on bar count (mostly the per-bar ONNX inference).

In [ ]:
inputs = dict(DEFAULT_INPUTS)
inputs.update(dict(
    base_lot            = LOT,
    spread_usd          = SPREAD_USD,
    use_variable_spread = USE_VARIABLE_SPREAD,
    max_stack           = MAX_STACK,
))
results = run_backtest(m5, sess, in_name, spec, inputs, deposit=DEPOSIT, verbose=True)
s = results['summary']
print()
print('=' * 70)
print(f'  RESULT')
print('=' * 70)
for k, v in s.items():
    print(f'  {k:18}: {v}')
print('=' * 70)
print(f'  Reminder: Python ~5pp more optimistic than MT5 Tester (no per-bar slippage).')


## Cell 7 â€” Equity curve + drawdown plot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
eq = np.array(results['equity_curve'], dtype=np.float64)
# replace warmup zeros with the deposit so the chart starts at the deposit
eq[eq == 0] = DEPOSIT
peak = np.maximum.accumulate(eq)
dd = peak - eq
dd_pct = dd / peak * 100
times = m5['time']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                                gridspec_kw={'height_ratios': [3, 1]})
ax1.plot(times, eq, lw=1.0, color='#0066cc', label='Equity')
ax1.plot(times, peak, lw=0.7, color='#999', label='Running peak', alpha=0.6)
ax1.axhline(DEPOSIT, ls=':', color='gray', label='Deposit')
ax1.set_ylabel('Equity (USD)')
ax1.set_title(f'MT5bot_m4Gold v1.30 - {s["return_pct"]:+.2f}%, PF {s["profit_factor"]}, '
              f'{s["n_trades"]} trades, max DD {s["max_dd_pct"]:.1f}%')
ax1.legend(loc='upper left'); ax1.grid(True, alpha=0.3)

ax2.fill_between(times, 0, -dd_pct, color='#cc0000', alpha=0.4)
ax2.set_ylabel('Drawdown (%)')
ax2.set_xlabel('Date')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Cell 8 â€” Per-year breakdown (regime survey)

In [ ]:
trades_df = pd.DataFrame(results['trades'])
if len(trades_df):
    trades_df['open_time'] = pd.to_datetime(trades_df['open_time'])
    trades_df['year'] = trades_df['open_time'].dt.year
    yearly = trades_df.groupby('year').agg(
        trades   = ('pnl', 'count'),
        wins     = ('pnl', lambda s: (s > 0).sum()),
        win_rate = ('pnl', lambda s: f'{(s > 0).mean()*100:.1f}%'),
        pnl_sum  = ('pnl', 'sum'),
        pf       = ('pnl', lambda s: f'{s[s>0].sum() / max(-s[s<=0].sum(), 1e-9):.3f}'),
        best     = ('pnl', 'max'),
        worst    = ('pnl', 'min'),
    ).round(2)
    print(yearly.to_string())
else:
    print('no trades')

## Cell 9 â€” Exit-reason histogram (sanity check)

In [ ]:
if len(trades_df):
    by_reason = trades_df.groupby('exit_reason').agg(
        n        = ('pnl', 'count'),
        win_rate = ('pnl', lambda s: f'{(s > 0).mean()*100:.1f}%'),
        avg_pnl  = ('pnl', 'mean'),
        sum_pnl  = ('pnl', 'sum'),
    ).round(2)
    print(by_reason.to_string())
    print()
    print('Expected breakdown for a healthy trend-follower with breakeven:')
    print('  - trend-flip exits should be the largest BY $ (the winners)')
    print('  - SL hits should be the largest BY COUNT (the small losers)')
    print('  - timeout exits should be a minority')

## Cell 10 â€” Stress-fold view
Slice the test into 6-month sub-periods (matching the deploy-gate `min_fold_pf` test) and check if any sub-period catastrophically loses.

In [ ]:
if len(trades_df):
    trades_df['period'] = trades_df['open_time'].dt.to_period('6M').astype(str)
    fold = trades_df.groupby('period').agg(
        trades  = ('pnl', 'count'),
        sum_pnl = ('pnl', 'sum'),
        pf      = ('pnl', lambda s: round(s[s>0].sum() / max(-s[s<=0].sum(), 1e-9), 3)),
    ).round(2)
    print(fold.to_string())
    print()
    bad_folds = (fold['pf'] < 1.0).sum()
    total = len(fold)
    print(f'  STRESS GATE: {total - bad_folds}/{total} folds positive (PF >= 1.0)')
    print(f'  Deploy gate (live EA) requires no fold worse than PF 0.92.')
    worst_pf = fold['pf'].min()
    print(f'  Worst fold PF: {worst_pf}  -- {"PASS" if worst_pf >= 0.92 else "FAIL"}')

## Cell 11 â€” Save results back to /kaggle/working
(Optional - so you can download them or use them in another notebook)

In [ ]:
import json
out = dict(results)
out.pop('equity_curve')   # too big for JSON
with open('/kaggle/working/v130_backtest_results.json', 'w') as f:
    json.dump(out, f, indent=2, default=str)
trades_df.to_csv('/kaggle/working/v130_trades.csv', index=False)
print('saved /kaggle/working/v130_backtest_results.json')
print('saved /kaggle/working/v130_trades.csv')